# Initial Setup
This notebook guides you through the initial setup steps required to use the GPUPhot system.

## Step 1: Verify Environment Variables
Before proceeding, ensure that the required environment variables are correctly configured. Run the following code to check the current values of the static environment variables:

In [ ]:
import os

# Default values for environment variables
default_values = {
    'ASTROMETRY_CACHE_PATH': './astrometry_cache',
    'INSTRUMENT_CONFIG_PATH': './gpuphot/instrument_configs',
    'IMAGE_PATH': '~/gpuphot_images',
}

# Check static environment variables
print('Current environment variable values (or defaults if not defined):\n')
for var, default in default_values.items():
    value = os.getenv(var, default)
    print(f'{var}: {value}')

# Check if any variable is using the default value
using_defaults = [var for var, default in default_values.items() if os.getenv(var) is None]
if using_defaults:
    print(f'\nNote: The following variables are using default values: {using_defaults}')
else:
    print('\nAll required environment variables are defined in the .env file.')

## Step 2: Download Astrometry Index Files
Before processing astronomical images, you need to download the astrometry index files. These files are required for the astrometric calibration of the images.

**Important:** This process uses the `ASTROMETRY_CACHE_PATH` variable from your `.env` file to mount a directory from your host machine. The code below will confirm the exact host path being used and then begin the download.

Run the following code to download the index files:

In [ ]:
import os
from gpuphot.utils.astro import get_solver

def download_astrometry_files():
    """
    Checks for the astrometry cache path configuration and triggers the download
    of the index files, providing clear information to the user about the host and
    container paths.
    """
    # This is the path INSIDE the Docker container, where the application runs.
    internal_cache_path = '/data/astrometry_cache'

    # This environment variable is passed from the docker-compose.yml
    # to make the HOST path visible inside the container for informational purposes.
    host_cache_path = os.getenv('ASTROMETRY_CACHE_PATH_HOST')

    if host_cache_path:
        print("--- Astrometry Index File Setup ---")
        print(f"The solver will download files to the directory mounted inside this container.")
        print(f"  - Container Path: {internal_cache_path}")
        print(f"  - Host Machine Path: {host_cache_path}\n")

        print("IMPORTANT:")
        print("The download process requires approximately 34GB of disk space and can take")
        print("over an hour, depending on your internet connection. This only needs to be done once.\n")

        try:
            # The get_solver() function is already configured to find the
            # cache at the internal path, which is '/data/astrometry_cache'.
            get_solver()
            print("--- Success ---")
            print(f"Astrometry index files are now available and stored on your host machine at: {host_cache_path}")
        except Exception as e:
            print(f"--- Error ---")
            print(f"An error occurred during the solver initialization: {e}")
            print("Please check your internet connection, disk space, and file permissions on the host path.")

    else:
        print("--- Configuration Error ---")
        print("Error: The 'ASTROMETRY_CACHE_PATH_HOST' environment variable is not set inside the container.")
        print("Please ensure it is correctly defined in your docker-compose.yml for the 'lab' service.")
        print("Example for docker-compose.yml:")
        print("  services:")
        print("    lab:")
        print("      environment:")
        print("        - ASTROMETRY_CACHE_PATH_HOST=${ASTROMETRY_CACHE_PATH:-./astrometry_cache}")


# To run the process, simply call the function in your notebook cell.
download_astrometry_files()


## Step 3: Configure Environment Variables
The GPUPhot system relies on several environment variables to function correctly. These variables control settings such as database connections, file paths, and logging.

### Example `.env` File
Below is an example of a `.env` file with all the available environment variables. You can copy this template and modify it according to your setup.

```bash
# Library settings
GPUPHOT_DEBUG=True  # Enable debug mode for detailed logging
GPUPHOT_LOG_LEVEL=DEBUG  # Logging level (DEBUG, INFO, WARNING, ERROR, CRITICAL)
GPUPHOT_ENVIRONMENT=development  # Environment (development, production)

# Astrometry settings
ASTROMETRY_CACHE_PATH=/path/to/astrometry_cache  # Path on the host to store astrometry index files

# Worker settings
INSTRUMENT_NAME=default  # Default instrument configuration
INSTRUMENT_CONFIG_PATH=/path/to/instrument_configs  # Path on the host to instrument configuration files
IMAGE_PATH=/path/to/images  # Base path on the host for astronomical images

# Logging settings
LOGSTASH_LOGGING=True  # Enable Logstash logging (optional)
LOGSTASH_HOST=10.10.10.10  # Logstash server IP or hostname
LOGSTASH_PORT=5000  # Logstash server port

# Database settings
POSTGRES_PASSWORD=gpuphot  # Password for the PostgreSQL database
```

### Instructions
1. Create a file named `.env` in the root directory of your project.
2. Copy the example above into the `.env` file.
3. Replace the placeholder paths and values with your actual configuration.

**Note:** The `.env` file is optional. If you don't provide one, the system will use default values for most settings.

**Important:** The paths specified in `ASTROMETRY_CACHE_PATH`, `INSTRUMENT_CONFIG_PATH`, and `IMAGE_PATH` must be paths on the **host machine**, not inside the Docker container. These paths will be mounted into the container at runtime.

## Next Steps
Once the setup is complete, you can proceed to the following notebooks:
- [2. Instrument Configuration](./2_Instrument_Configuration_Notebook.ipynb): Learn how to configure instruments and explore configuration files.
- [3. Task Execution](./3_Task_Execution_Notebook.ipynb): Learn how to process images using Celery tasks.
- [4. Database Query](./4_Database_Query_Notebook.ipynb): Explore the database and query photometric data.
